# 02a - Probability & Inference Analyst: Pemodelan Probabilitas  (Minggu 2)

**Peran:** Probability & Inference Analyst - *peluang bersyarat, distribusi, CI & uji hipotesis*
**Dikerjakan oleh:** _(isi nama)_

### Cerita singkat tugasmu
Kamu menjawab pertanyaan **"seberapa mungkin?"** dari data. Contoh: *"Kalau lagi jam sibuk, seberapa mungkin udara jadi Tidak Sehat?"*
Minggu ini ada 3 bagian:

| Bagian | Isi | Dari panduan dosen |
|---|---|---|
| A | Peluang empiris, minimal 3 kejadian, minimal 2 di antaranya **peluang bersyarat** | Minggu 2a |
| B | Uji kecocokan 1 variabel kontinu dengan **Distribusi Normal** + hitung peluang 2 skenario | Minggu 2b |
| C | 1 variabel biner -> **Distribusi Binomial** n = 20: hitung P(X = k) dan P(X >= k) | Minggu 2c |

> ✏️ **ISI** = bagian yang harus kamu tulis dengan kata-katamu sendiri. Ingat: saat Gallery Walk, kamulah yang harus menjelaskan bagian ini.
>
> ⚠️ Pastikan data bersih resmi dari Data Engineer sudah ada. Kalau belum, kamu akan melihat peringatan DRAF SEMENTARA - boleh berlatih dulu, tapi **jalankan ulang semua sel** setelah data resmi masuk.

In [ ]:
# ============ SETUP (jalankan sel ini PERTAMA) ============
# Satu-satunya yang boleh diubah: GITHUB_USER dan GITHUB_REPO (diisi sekali oleh Project Lead).
import os, sys, subprocess
GITHUB_USER = "Tur-ai"      # <- username GitHub pemilik repo
GITHUB_REPO = "udara-mobilitas-statsprob"     # <- nama repository
BRANCH = "main"

if "google.colab" in sys.modules:                      # kalau dijalankan di Google Colab
    PROYEK = "/content/proyek"
    if not os.path.exists(PROYEK):
        token = ""
        try:                                           # opsional: untuk repo PRIVATE (lihat docs/02_...)
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN") + "@"
        except Exception:
            pass
        url = f"https://{token}github.com/{GITHUB_USER}/{GITHUB_REPO}.git"
        r = subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, PROYEK],
                           capture_output=True, text=True)
        if r.returncode != 0:
            pesan = (r.stderr or "").replace(token.rstrip("@"), "***") if token else (r.stderr or "")
            raise RuntimeError("Gagal clone repo. Cek GITHUB_USER/GITHUB_REPO, atau lihat bagian "
                               "'Troubleshooting' di docs/02_panduan_github_colab.md\n" + pesan)
    else:
        subprocess.run(["git", "-C", PROYEK, "pull"], capture_output=True)   # ambil update terbaru
    os.chdir(PROYEK)
elif os.path.basename(os.getcwd()) == "notebooks":    # kalau dijalankan di Jupyter lokal
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
from src.helpers import *
sns.set_theme(style="whitegrid", font_scale=1.0)
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
print("Setup selesai. Folder kerja:", os.getcwd())

In [ ]:
df = muat_data_bersih()
N = len(df)
df.head()

## A. Peluang empiris dan peluang bersyarat (Minggu 2a)

**Peluang empiris** = peluang yang dihitung dari data yang benar-benar terjadi:

$$P(A) = \frac{\text{jumlah baris yang memenuhi A}}{\text{total baris}}$$

**Peluang bersyarat** = peluang A **jika kita sudah tahu B terjadi**. Kita "mempersempit" data hanya ke baris B, lalu hitung berapa persen di antaranya yang A:

$$P(A \mid B) = \frac{P(A \cap B)}{P(B)} = \frac{\text{jumlah baris A dan B}}{\text{jumlah baris B}}$$

Baca `P(A | B)` sebagai **"peluang A, dengan syarat B"**. Contoh: P(Tidak Sehat | Jam Sibuk) = dari semua pengukuran saat jam sibuk, berapa persen yang udaranya Tidak Sehat.

In [ ]:
# Definisikan kejadian sebagai "filter" True/False untuk tiap baris
tidak_sehat  = df["Status_Udara"] == "Tidak Sehat"
sedang       = df["Status_Udara"] == "Sedang"
baik         = df["Status_Udara"] == "Baik"
jam_sibuk    = df["Jam_Sibuk"] == "Ya"
bukan_sibuk  = df["Jam_Sibuk"] == "Tidak"
hari_kerja   = df["Jenis_Hari"] == "Hari Kerja"
pusat_kota   = df["Zona_Lokasi"] == "Pusat Kota"
pinggiran    = df["Zona_Lokasi"] == "Pinggiran"

def peluang(A, nama):
    """P(A)"""
    n = int(A.sum())
    return {"Kejadian": nama, "Rumus": f"{n} / {N}", "Peluang": n / N}

def peluang_bersyarat(A, B, nama):
    """P(A | B)"""
    n_ab, n_b = int((A & B).sum()), int(B.sum())
    return {"Kejadian": nama, "Rumus": f"{n_ab} / {n_b}", "Peluang": n_ab / n_b}

hasil = [
    # --- peluang biasa (minimal 3) ---
    peluang(tidak_sehat, "P(Tidak Sehat)"),
    peluang(jam_sibuk,   "P(Jam Sibuk)"),
    peluang(pusat_kota,  "P(Pusat Kota)"),
    peluang(hari_kerja & jam_sibuk, "P(Hari Kerja dan Jam Sibuk)"),
    # --- peluang bersyarat (minimal 2) ---
    peluang_bersyarat(tidak_sehat, jam_sibuk,   "P(Tidak Sehat | Jam Sibuk)"),
    peluang_bersyarat(tidak_sehat, bukan_sibuk, "P(Tidak Sehat | Bukan Jam Sibuk)"),
    peluang_bersyarat(jam_sibuk,   tidak_sehat, "P(Jam Sibuk | Tidak Sehat)"),
    peluang_bersyarat(tidak_sehat, pusat_kota,  "P(Tidak Sehat | Pusat Kota)"),
    peluang_bersyarat(tidak_sehat, pinggiran,   "P(Tidak Sehat | Pinggiran)"),
]
tabel_peluang = pd.DataFrame(hasil)
tabel_peluang["Peluang"] = tabel_peluang["Peluang"].round(4)
tabel_peluang["Persen"] = (tabel_peluang["Peluang"] * 100).round(2)
simpan_tabel(tabel_peluang, "02_peluang_empiris", index=False)
tabel_peluang

### Perhatikan: P(A | B) tidak sama dengan P(B | A)!
Bandingkan `P(Tidak Sehat | Jam Sibuk)` dengan `P(Jam Sibuk | Tidak Sehat)`. Yang pertama menjawab *"saat jam sibuk, seberapa sering udara buruk?"*, yang kedua *"saat udara buruk, seberapa sering itu jam sibuk?"*. Pertanyaannya beda, angkanya beda.

### Apakah "Tidak Sehat" dan "Jam Sibuk" saling bebas (independen)?
Dua kejadian **independen** kalau mengetahui B tidak mengubah peluang A, artinya `P(A | B) = P(A)`.

In [ ]:
p_a = tidak_sehat.mean()
p_a_given_b = (tidak_sehat & jam_sibuk).sum() / jam_sibuk.sum()
print(f"P(Tidak Sehat)              = {p_a:.4f}")
print(f"P(Tidak Sehat | Jam Sibuk)  = {p_a_given_b:.4f}")
print(f"Rasio P(A|B) / P(A)         = {p_a_given_b / p_a:.2f}   (1.00 berarti persis independen)")

✏️ **ISI - Interpretasi (bahasa sehari-hari, 2-3 kalimat):**
_Contoh kerangka: "Dari ... pengukuran, ...% berstatus Tidak Sehat. Saat jam sibuk, angkanya menjadi ...%, sedangkan di luar jam sibuk ...%. Artinya jam sibuk (menaikkan / tidak banyak mengubah) peluang udara tidak sehat."_

_(tulis di sini)_

⚠️ **Hati-hati:** selisih kecil belum tentu "nyata" - bisa saja kebetulan. Pembuktiannya nanti lewat uji Chi-Square di Minggu 3.

## B. Distribusi Normal (Minggu 2b)

**Distribusi Normal** = bentuk "lonceng" simetris. Ditentukan 2 angka: rata-rata $\mu$ (pusat) dan simpangan baku $\sigma$ (lebar).
Kita pilih **`Indeks_PM25`** (variabel utama) lalu menjawab dua hal:
1. Apakah datanya *cukup mirip* lonceng normal?
2. Kalau kita anggap normal, berapa peluang PM2.5 melewati batas tertentu?

Kita memakai $\mu$ dan $\sigma$ **dari sampel**: `mean()` dan `std()` (pandas otomatis memakai n-1).

In [ ]:
VAR = "Indeks_PM25"
x = df[VAR]
mu, sigma = x.mean(), x.std(ddof=1)
print(f"mean (mu)    = {mu:.2f}")
print(f"std (sigma)  = {sigma:.2f}")
print(f"skewness     = {x.skew():.2f}   (0 = simetris, >0 = ekor panjang ke kanan, <0 = ke kiri)")

### B1. Uji kecocokan secara visual & numerik
Tiga alat:
1. **Histogram + kurva normal** - apakah bentuknya mirip lonceng?
2. **Q-Q plot** - kalau titik-titik mengikuti garis lurus, data mendekati normal. Titik melengkung di ujung = ekor lebih panjang/pendek dari normal.
3. **Aturan empiris 68-95-99.7** - pada normal sungguhan, ~68% data berada dalam +-1 sigma, ~95% dalam +-2 sigma, ~99.7% dalam +-3 sigma.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Histogram + kurva normal
sns.histplot(x, bins=25, stat="density", color="#9ecae1", edgecolor="white", ax=axes[0])
grid = np.linspace(x.min(), x.max(), 300)
axes[0].plot(grid, stats.norm.pdf(grid, mu, sigma), color="#c0392b", lw=2, label=f"Normal(mu={mu:.1f}, sigma={sigma:.1f})")
axes[0].set_title("Histogram PM2.5 vs kurva normal"); axes[0].set_xlabel(VAR); axes[0].legend()

# Q-Q plot
stats.probplot(x, dist="norm", plot=axes[1])
axes[1].set_title("Q-Q plot PM2.5")
plt.tight_layout()
simpan_gambar(fig, "02_normal_histogram_qq")
plt.show()

In [ ]:
# Aturan empiris: berapa persen data yang BENAR-BENAR berada dalam +-k sigma?
baris = []
for k, teori in [(1, 68.27), (2, 95.45), (3, 99.73)]:
    aktual = ((x >= mu - k * sigma) & (x <= mu + k * sigma)).mean() * 100
    baris.append([f"mu +- {k} sigma", teori, round(aktual, 2)])
tabel_empiris = pd.DataFrame(baris, columns=["Rentang", "Teori_Normal_%", "Data_Sebenarnya_%"])
simpan_tabel(tabel_empiris, "02_normal_aturan_empiris", index=False)
tabel_empiris

In [ ]:
# Uji formal Shapiro-Wilk. H0: data berasal dari distribusi normal.
w, p_shapiro = stats.shapiro(x)
print(f"Shapiro-Wilk: W = {w:.4f}, p-value = {p_shapiro:.4g}")
print("Keputusan (alpha = 0.05):", "TOLAK H0 -> data menyimpang dari normal" if p_shapiro < ALPHA else "GAGAL TOLAK H0 -> tidak ada bukti menyimpang dari normal")

✏️ **ISI - Kesimpulan kecocokan normal (3-4 kalimat):**
Pertimbangkan: bentuk histogram, kelurusan Q-Q plot, skewness, kecocokan aturan empiris, dan p-value Shapiro.
Catatan penting: pada sampel besar (n > 300), Shapiro-Wilk **sangat sensitif** - penyimpangan kecil pun bisa menghasilkan p kecil. Jadi jangan hanya mengandalkan p-value; lihat juga grafiknya.

_(tulis di sini: "Menurut kami PM2.5 (cukup mendekati / kurang mendekati) normal karena ...")_

### B2. Hitung peluang untuk 2 skenario
Kita ubah nilai batas menjadi **skor Z** ("berapa simpangan baku dari rata-rata"):

$$Z = \frac{x - \mu}{\sigma}$$

lalu cari luas area di bawah kurva normal dengan `stats.norm.cdf(z)`  (cdf = peluang nilai **kurang dari atau sama dengan** itu).

Dua skenario yang bermakna dari data kita (batas kategori status udara):
- **Skenario 1:** P(PM2.5 > 65) -> peluang masuk kategori *Tidak Sehat*
- **Skenario 2:** P(PM2.5 <= 35) -> peluang masuk kategori *Baik*

✏️ Boleh ganti batasnya dengan angka lain yang menurut tim lebih bermakna.

In [ ]:
def peluang_normal(batas, arah):
    z = (batas - mu) / sigma
    if arah == "lebih_dari":
        p = 1 - stats.norm.cdf(z)
        emp = (x > batas).mean()
        teks = f"P(X > {batas})"
    else:
        p = stats.norm.cdf(z)
        emp = (x <= batas).mean()
        teks = f"P(X <= {batas})"
    return {"Skenario": teks, "Z": round(z, 3), "Peluang_Normal": round(p, 4), "Proporsi_Data_Sebenarnya": round(emp, 4)}

tabel_normal = pd.DataFrame([
    peluang_normal(65, "lebih_dari"),
    peluang_normal(35, "kurang_dari"),
])
tabel_normal["Selisih"] = (tabel_normal["Peluang_Normal"] - tabel_normal["Proporsi_Data_Sebenarnya"]).round(4)
simpan_tabel(tabel_normal, "02_normal_peluang", index=False)
tabel_normal

In [ ]:
# Gambar area peluangnya (untuk laporan/poster)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
grid = np.linspace(mu - 4 * sigma, mu + 4 * sigma, 400)
pdf = stats.norm.pdf(grid, mu, sigma)

axes[0].plot(grid, pdf, color="black")
axes[0].fill_between(grid, pdf, where=grid > 65, color="#d64545", alpha=0.6)
axes[0].axvline(65, ls="--", color="gray"); axes[0].set_title("Skenario 1: P(PM2.5 > 65)")

axes[1].plot(grid, pdf, color="black")
axes[1].fill_between(grid, pdf, where=grid <= 35, color="#2e9e5b", alpha=0.6)
axes[1].axvline(35, ls="--", color="gray"); axes[1].set_title("Skenario 2: P(PM2.5 <= 35)")
for a in axes: a.set_xlabel("Indeks PM2.5"); a.set_yticks([])
plt.tight_layout()
simpan_gambar(fig, "02_normal_area_peluang")
plt.show()

✏️ **ISI - Interpretasi:**
1. Berapa peluang menurut model normal, dan berapa proporsi sebenarnya di data?
2. Apakah selisihnya kecil atau besar? Kalau besar, apa artinya bagi cocok-tidaknya model normal? (petunjuk: lihat Q-Q plot & skewness; data PM2.5 tidak bisa di bawah 0, sedangkan kurva normal memperbolehkan nilai negatif)

_(tulis di sini)_

## C. Distribusi Binomial (Minggu 2c)

**Distribusi Binomial** dipakai ketika kita melakukan **n percobaan yang sama**, tiap percobaan hanya punya 2 hasil (**sukses / gagal**) dengan peluang sukses **p yang tetap**, dan antar percobaan tidak saling mempengaruhi.
$X$ = jumlah sukses dari n percobaan.

$$P(X = k) = \binom{n}{k} p^k (1-p)^{n-k}$$

**Cerita:** bayangkan petugas mengukur udara di **20 titik acak**. $X$ = berapa titik yang statusnya *Tidak Sehat*.
- "Sukses" = status Tidak Sehat (di sini "sukses" hanya istilah teknis, bukan berarti bagus)
- $p$ = proporsi Tidak Sehat di data
- $n = 20$ (ditetapkan dosen)

In [ ]:
# Variabel biner: 1 = Tidak Sehat (sukses), 0 = bukan
sukses = (df["Status_Udara"] == "Tidak Sehat")
p = sukses.mean()
n = 20
print(f"Proporsi sukses p = {sukses.sum()} / {N} = {p:.4f}")
print(f"Ekspektasi E(X) = n*p = {n*p:.2f} titik Tidak Sehat dari {n}")
print(f"Simpangan baku  = sqrt(n*p*(1-p)) = {np.sqrt(n*p*(1-p)):.2f}")

In [ ]:
# Tabel lengkap P(X = k) dan P(X >= k) untuk k = 0..20
k = np.arange(0, n + 1)
tabel_binom = pd.DataFrame({
    "k": k,
    "P(X = k)": stats.binom.pmf(k, n, p),
    "P(X <= k)": stats.binom.cdf(k, n, p),
    "P(X >= k)": stats.binom.sf(k - 1, n, p),     # sf(k-1) = 1 - cdf(k-1) = P(X >= k)
}).round(4)
simpan_tabel(tabel_binom, "02_binomial_tabel", index=False)
tabel_binom.head(10)

In [ ]:
# Pertanyaan yang diminta: pilih k tertentu
K_TEPAT, K_MINIMAL = 2, 3          # ✏️ boleh diganti sesuai pertanyaan yang tim anggap menarik
p_tepat   = stats.binom.pmf(K_TEPAT, n, p)
p_minimal = stats.binom.sf(K_MINIMAL - 1, n, p)
print(f"P(X = {K_TEPAT})  = {p_tepat:.4f}  ({p_tepat*100:.1f}%)")
print(f"P(X >= {K_MINIMAL}) = {p_minimal:.4f}  ({p_minimal*100:.1f}%)")

fig, ax = plt.subplots(figsize=(8, 4))
warna = ["#d64545" if i >= K_MINIMAL else "#9ecae1" for i in k]
ax.bar(k, stats.binom.pmf(k, n, p), color=warna)
ax.set_xlabel("k = jumlah titik 'Tidak Sehat' dari 20 titik"); ax.set_ylabel("P(X = k)")
ax.set_title(f"Binomial(n=20, p={p:.3f}) - merah = area P(X >= {K_MINIMAL})")
ax.set_xticks(range(0, 13)); ax.set_xlim(-0.7, 12.7)
plt.tight_layout()
simpan_gambar(fig, "02_binomial_pmf")
plt.show()

✏️ **ISI - Interpretasi (bahasa sehari-hari):**
_Kerangka: "Jika kita mengukur 20 titik secara acak, peluang tepat ... titik berstatus Tidak Sehat adalah ...%. Peluang ada **paling sedikit** ... titik Tidak Sehat adalah ...%. Artinya ..."_

_(tulis di sini)_

**Cek asumsi (jawab singkat):** apakah tiap pengukuran bisa dianggap independen dan p-nya sama? Kalau data diambil di jam & lokasi yang berbeda-beda, p sebenarnya tidak persis sama - model binomial adalah **pendekatan**. Sebutkan ini di laporan sebagai keterbatasan.

## Serahkan hasil
Jalankan sel di bawah -> unduh ZIP -> upload ke GitHub (panduan: `docs/02_panduan_github_colab.md`).

In [ ]:
unduh_hasil("hasil_02a_probability")

### ✅ Checklist selesai (Minggu 2)
- [ ] >= 3 peluang empiris, >= 2 di antaranya bersyarat, interpretasi sudah ditulis
- [ ] Histogram + Q-Q plot + Shapiro + kesimpulan normal tertulis
- [ ] 2 skenario peluang normal + gambar area + interpretasi
- [ ] Binomial n=20: P(X=k) dan P(X>=k) + grafik + interpretasi
- [ ] Hasil sudah di-upload ke `outputs/` dan dikabarkan ke Technical Writer & Project Lead
- [ ] Aku bisa menjelaskan tanpa membaca: apa itu peluang bersyarat, apa itu skor Z, kapan binomial dipakai